# Tutorial: Project Question, Data, and the Explanatory Mainline

**Audience**
- Cross-disciplinary readers who have not seen this project before.

**Prerequisites**
- Basic Python reading skills.
- A rough idea of what a model, a feature, and a validation metric are.

**Learning goals**
- Understand the research question behind the repository.
- See how raw event rasters become a pixel-level modeling panel.
- Identify the core explanatory model family and its strongest result.


## Outline

1. Why this project exists.
2. A visual intuition before the models.
3. How the pixel panel is built.
4. How the explanatory models are fitted.
5. What feature-upgrade and strict-v2 add.
6. Which stored figures already tell the story.


In [ ]:
from __future__ import annotations

import ast
import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
MODELING_DIR = REPO_ROOT / "project" / "modeling"
OUTPUT_DIR = MODELING_DIR / "output"


def load_json(rel_path: str):
    return json.loads((REPO_ROOT / rel_path).read_text(encoding="utf-8"))


def load_csv(rel_path: str) -> pd.DataFrame:
    return pd.read_csv(REPO_ROOT / rel_path)


def get_def_source(rel_path: str, name: str, max_lines: int = 80) -> str:
    source = (REPO_ROOT / rel_path).read_text(encoding="utf-8")
    tree = ast.parse(source)
    lines = source.splitlines()
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
            block = lines[node.lineno - 1 : node.end_lineno]
            if len(block) > max_lines:
                block = block[:max_lines] + ["# ... truncated for notebook readability ..."]
            return "\n".join(block)
    raise KeyError(f"{name} was not found in {rel_path}")


def print_defs(rel_path: str, *names: str, max_lines: int = 80) -> None:
    for name in names:
        print(f"\n===== {name} ({rel_path}) =====\n")
        print(get_def_source(rel_path, name, max_lines=max_lines))


print("Repository root:", REPO_ROOT)
print("Modeling directory:", MODELING_DIR)


## 1. Why this project exists

The active modeling code lives in:

- `project/modeling/pipeline_lib.py`
- `project/modeling/pipelines/01_in_sample_pipeline.py`
- `project/modeling/pipelines/02_cross_event_pipeline.py`
- `project/modeling/pipelines/03_exploration_pipeline.py`

The project asks a simple but demanding question:

> After a disaster, do pixels near critical infrastructure behave as if backup power is present?

That question is only answerable after the repo converts rasters, POI buffers, land-use controls, and event labels into one coherent pixel panel.


In [ ]:
events6 = load_json('project/modeling/config/events_6.json')
events10 = load_json('project/modeling/config/events_10.json')

events6 = pd.DataFrame.from_dict(events6, orient='index')
events10 = pd.DataFrame.from_dict(events10, orient='index')

display(events6[['event_name', 'metric_crs', 'pre_dir', 'post_dir']])
display(events10[['event_name', 'disaster_type', 'country_scope']])


## 2. A visual intuition before the models

Before reading code, it helps to remember that the project starts with nightlight changes over time, not with a ready-made table.

The figure below is one raw example of a nightlight time series already stored in the repo.


![](../../result/plots_eventkey/Harris_Texas-3_Harris_201708281215_201709011700_mean_ntl_timeseries.png)

## 3. How the pixel panel is built

In [ ]:
print_defs(
    'project/modeling/pipeline_lib.py',
    'build_pixel_panel',
    'attach_nlcd',
    max_lines=80,
)


The panel builder creates the variables that appear again and again throughout the project:

- `pre_mean_ntl`, `post_mean_ntl`, `delta_ntl`
- `in_buffer`, `distance_to_nearest`, `n_facilities_in_buffer`
- event IDs, pixel IDs, and later land-use / cloud controls


## 4. How the explanatory models are fitted

In [ ]:
print_defs(
    'project/modeling/pipeline_lib.py',
    'fit_ols_and_mixed',
    'fit_logit',
    'build_recovery_panel',
    'fit_cox',
    max_lines=80,
)


## 5. What feature-upgrade and strict-v2 add

In [ ]:
print_defs(
    'project/modeling/pipelines/01_in_sample_pipeline.py',
    'cmd_full_run',
    'build_parser',
    'main',
    max_lines=70,
)


## Current strict-v2 reading

- OLS `coef_in_buffer = 0.0269` with `p = 0.0544`.
- MixedLM `coef_in_buffer = 0.0269` with `p = 0.0094`.
- Logit `odds_ratio_in_buffer = 0.7503` and LOEO `AUC = 0.4549`.
- Cox `hazard_ratio_in_buffer = 1.3319` and LOEO `c_index = 0.5200`.

Read this notebook as: the explanatory signal is stable, but out-of-event transport is already difficult.


## 6. Key figures already stored in the repository

In [ ]:
strict_summary = load_csv('project/modeling/output/model_summary_feature_upgrade_v2_strict.csv')
logo = load_csv('project/modeling/output/logo_aggregate_metrics_v2_strict.csv')

display(strict_summary[['model', 'variant', 'key_metric', 'value', 'p_value']])
display(logo)


## Figures to read

### Feature-upgrade model comparison
![](../figures/feature_upgrade/feature_upgrade_model_compare_locked.png)

### Before/after comparison
![](../figures/feature_upgrade/model_compare_before_after.png)

### Leave-one-event-out aggregate metrics
![](../figures/logo/logo_aggregate_metrics.png)


## 7. What this notebook should leave you with

- The repository's explanatory heart is the pixel panel + four core model families.
- `strict-v2` is the explanatory anchor because it keeps the cohort locked while adding the strongest control set.
- `run_pipeline.py`, the root-level `15-19` files, and `legacy/` are not where the real modeling story lives.


## Optional reader exercise

Pick one event in `events_6.json` and explain how its raw rasters, POI file, and metric CRS connect to the final panel.


In [ ]:
# Exercise scaffold: replace the event_id and inspect its raw config.
event_id = 'ida_neworleans'
events6.loc[event_id]
